### Comparison of ODYSSEA SST and Oleander XBT Temperature Data (2024)

This notebook compares the adjusted_sea_surface_temperature from ODYSSEA with near-surface temperature measurements from 2024 XBT profiles

In [ ]:
from pprint import pprint
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import glob
import os

In [ ]:
# Load ODYSSEA dataset
ds_ODYSSEA = xr.open_dataset(
    "../odyssea_2024/cmems_obs-sst_glo_phy_my_l3s_P1D-m_multi-vars_84.95W-30.05W_15.05N-59.95N_2024-01-01-2024-12-31.nc"
)

print("ODYSSEA dataset loaded:")
print(ds_ODYSSEA)

In [ ]:
# Find XBT files
xbt_files = sorted(glob.glob("../xbt_2024/*.nc"))

print(f"Found {len(xbt_files)} XBT files:")
for f in xbt_files:
    print(f"  - {os.path.basename(f)}")

Note: Both datasets use UTC timestamps (ODYSSEA nc file do not state it explicitly but follow CF convetion, XBT files have 'Z' suffix in time values)

In [ ]:
ds0 = xr.open_dataset(xbt_files[0], engine='netcdf4')

In [ ]:
ds0

In [ ]:
# Process all XBT files and extract comparison data
xbt_lats = []
xbt_lons = []
xbt_times = []
xbt_surface_temps = []
xbt_depths_used = []
odyssea_lats = []
odyssea_lons = []
odyssea_temps = []
odyssea_times = []
temp_diffs = []
xbt_filenames = []  # Track which file each point came from

def distance_to_depth_range(depth, min_depth=1.0, max_depth=5.0):
    """Calculate distance from depth to [min_depth, max_depth] range"""
    if min_depth <= depth <= max_depth:
        return 0.0
    elif depth < min_depth:
        return min_depth - depth
    else:
        return depth - max_depth

# Iterate through each XBT file
for xbt_file in xbt_files:
    print(f"Processing {os.path.basename(xbt_file)}...")
    
    # Load XBT dataset
    ds_XBT = xr.open_dataset(xbt_file, engine='netcdf4')
    
    # Iterate through each profile in this file
    for profile_idx in range(ds_XBT.dims['profile']):
        # Get XBT profile data (lat, lon, time have (trajectory, profile) dimensions)
        xbt_lat = float(ds_XBT.latitude[0, profile_idx].values)
        xbt_lon = float(ds_XBT.longitude[0, profile_idx].values)
        xbt_time = ds_XBT.time[0, profile_idx].values
        
        # Get temperature profile and depth (temp and depth have (trajectory, profile, z) dimensions)
        temp_profile = ds_XBT.temp[0, profile_idx, :].values
        depth_profile = ds_XBT.depth[0, profile_idx, :].values
        
        # Remove nans
        valid_mask = ~np.isnan(temp_profile) & ~np.isnan(depth_profile)
        if not valid_mask.any():
            continue
        valid_depths = depth_profile[valid_mask]
        valid_temps = temp_profile[valid_mask]
        
        # Find measurement closest to 1-5m depth range (depth of ODYSSEA measurements)
        distances = np.array([distance_to_depth_range(d) for d in valid_depths])
        best_idx = np.argmin(distances) # measurement closest to 1-5m range
        xbt_depth = valid_depths[best_idx]
        xbt_temp = valid_temps[best_idx]
        
        # Find closest ODYSSEA measurement in space and time
        odyssea_point = ds_ODYSSEA['adjusted_sea_surface_temperature'].sel(
            latitude=xbt_lat,
            longitude=xbt_lon,
            time=xbt_time,
            method='nearest'
        )
        
        # Get the coordinates for selected ODYSSEA measurement
        odyssea_lat = float(odyssea_point.latitude.values)
        odyssea_lon = float(odyssea_point.longitude.values)
        odyssea_temp = float(odyssea_point.values)
        odyssea_time = odyssea_point.time.values
        
        # Skip if ODYSSEA data is NaN
        if np.isnan(odyssea_temp):
            continue
        
        # Convert ODYSSEA from Kelvin to Celsius
        odyssea_temp_celsius = odyssea_temp - 273.15
        
        # Calculate difference (XBT - ODYSSEA)
        # Note: if temp_diff > 0 (< 0), then XBT is warmer (colder) than ODYSSEA
        temp_diff = xbt_temp - odyssea_temp_celsius
        
        # Store results
        xbt_lats.append(xbt_lat)
        xbt_lons.append(xbt_lon)
        xbt_times.append(xbt_time)
        xbt_surface_temps.append(xbt_temp)
        xbt_depths_used.append(xbt_depth)
        odyssea_lats.append(odyssea_lat)
        odyssea_lons.append(odyssea_lon)
        odyssea_temps.append(odyssea_temp_celsius)
        odyssea_times.append(odyssea_time)
        temp_diffs.append(temp_diff)
        xbt_filenames.append(os.path.basename(xbt_file))
    
    ds_XBT.close()

# Convert to numpy arrays
xbt_lats = np.array(xbt_lats)
xbt_lons = np.array(xbt_lons)
xbt_times = np.array(xbt_times)
xbt_surface_temps = np.array(xbt_surface_temps)
xbt_depths_used = np.array(xbt_depths_used)
odyssea_lats = np.array(odyssea_lats)
odyssea_lons = np.array(odyssea_lons)
odyssea_temps = np.array(odyssea_temps)
odyssea_times = np.array(odyssea_times)
temp_diffs = np.array(temp_diffs)

print(f"\nTotal valid comparison points: {len(temp_diffs)}")
print(f"XBT depths used - min: {xbt_depths_used.min():.2f}m, max: {xbt_depths_used.max():.2f}m, mean: {xbt_depths_used.mean():.2f}m")
print(f"XBT depths in [1-5]m range: {np.sum((xbt_depths_used >= 1) & (xbt_depths_used <= 5))} / {len(xbt_depths_used)}")

In [ ]:
# Calculate and print statistics
mean_diff = np.mean(temp_diffs)
std_diff = np.std(temp_diffs)

# Calculate horizontal differences (XBT (lat,lon) - ODYSSEA (lat,lon))
xbt_points = np.stack((xbt_lons, xbt_lats), axis=1)
odyssea_points = np.stack((odyssea_lons, odyssea_lats), axis=1)
from geopy.distance import geodesic
km_diffs = [geodesic(p1, p2).km for p1, p2 in zip(xbt_points, odyssea_points)]

# Calculate depth differences (XBT depth - ODYSSEA nominal depth of 3m)
odyssea_nominal_depth = 3.0
depth_diffs = xbt_depths_used - odyssea_nominal_depth

# Calculate time differences in hours
time_diffs_seconds = (xbt_times - odyssea_times) / np.timedelta64(1, 's')
time_diffs_hours = time_diffs_seconds / 3600.0

print("=" * 60)
print("Temperature Comparison Statistics (XBT - ODYSSEA)")
print("=" * 60)
print(f"Mean difference: {mean_diff:.3f} °C")
print(f"Standard deviation: {std_diff:.3f} °C")
print(f"Min difference: {np.min(temp_diffs):.3f} °C")
print(f"Max difference: {np.max(temp_diffs):.3f} °C")
print(f"Number of comparisons: {len(temp_diffs)}")
print("=" * 60)
print("Horizontal Distance Statistics (XBT - ODYSSEA)")
print("=" * 60)
print(f"Mean horizontal distance: {np.mean(km_diffs):.3f} m")
print(f"Std horizontal distance: {np.std(km_diffs):.3f} m")
print(f"Min horizontal distance: {np.min(km_diffs):.3f} m")
print(f"Max horizontal distance: {np.max(km_diffs):.3f} m")
print("=" * 60)
print("=" * 60)
print("Depth Difference Statistics (XBT - ODYSSEA nominal 3m)")
print("=" * 60)
print(f"Mean depth difference: {np.mean(depth_diffs):.3f} m")
print(f"Std depth difference: {np.std(depth_diffs):.3f} m")
print(f"Min depth difference: {np.min(depth_diffs):.3f} m")
print(f"Max depth difference: {np.max(depth_diffs):.3f} m")
print("=" * 60)
print("Time Difference Statistics (XBT - ODYSSEA)")
print("=" * 60)
print(f"Mean time difference: {np.mean(time_diffs_hours):.3f} hours")
print(f"Std time difference: {np.std(time_diffs_hours):.3f} hours")
print(f"Min time difference: {np.min(time_diffs_hours):.3f} hours")
print(f"Max time difference: {np.max(time_diffs_hours):.3f} hours")
print("=" * 60)

In [ ]:
# Create map
fig = plt.figure(figsize=(14, 10))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()
ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)

# Normalize temperature differences for coloring
norm = Normalize(vmin=np.min(temp_diffs), vmax=np.max(temp_diffs))
cmap = plt.cm.RdBu_r  # Red for positive (XBT warmer), Blue for negative (ODYSSEA warmer)

# Plot XBT locations with colored markers
scatter = ax.scatter(
    xbt_lons,
    xbt_lats,
    c=temp_diffs,
    cmap=cmap,
    norm=norm,
    s=100,
    edgecolor='black',
    linewidth=1,
    transform=ccrs.PlateCarree(),
    zorder=5,
    label='XBT locations'
)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax, orientation='horizontal', pad=0.05, shrink=0.7)
cbar.set_label('Temperature Difference (XBT - ODYSSEA) [°C]', fontsize=12)

# Add title and legend
plt.title(
    f'XBT vs ODYSSEA SST Comparison - 2024\n'
    f'Mean Diff: {mean_diff:.2f}°C, Std: {std_diff:.2f}°C, N={len(temp_diffs)}',
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()
# plt.show()
plt.savefig('./oleander_odyssea_1.png')

In [ ]:
# Create histogram of temperature differences
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(temp_diffs, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(mean_diff, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_diff:.2f}°C')
ax.axvline(0, color='black', linestyle='-', linewidth=1, alpha=0.5)

ax.set_xlabel('Temperature Difference (XBT - ODYSSEA) [°C]', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title(f'Distribution of Temperature Differences\n(N={len(temp_diffs)}, σ={std_diff:.2f}°C)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
# plt.show()
plt.savefig('./oleander_odyssea_2.png')

In [ ]:
# Create scatter plot of XBT vs ODYSSEA temperatures
fig, ax = plt.subplots(figsize=(10, 10))

ax.scatter(odyssea_temps, xbt_surface_temps, alpha=0.5, s=50)

# Add 1:1 line
min_temp = min(odyssea_temps.min(), xbt_surface_temps.min())
max_temp = max(odyssea_temps.max(), xbt_surface_temps.max())
ax.plot([min_temp, max_temp], [min_temp, max_temp], 'r--', linewidth=2, label='1:1 line')

ax.set_xlabel('ODYSSEA SST [°C]', fontsize=12)
ax.set_ylabel('XBT Surface Temperature [°C]', fontsize=12)
ax.set_title(f'XBT vs ODYSSEA Temperature Comparison\nMean diff: {mean_diff:.2f}°C, RMSE: {np.sqrt(np.mean(temp_diffs**2)):.2f}°C', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

plt.tight_layout()
# plt.show()
plt.savefig('./oleander_odyssea_3.png')